# PhenoBench TensorFlow Object Detection Dataset Export

This notebook converts the :contentReference[oaicite:0]{index=0} dataset into TensorFlow Object Detection API compatible artifacts.

Generated outputs include:
- TFRecord datasets
- train/validation/test splits
- TensorFlow label maps
- representative dataset indices for INT8 quantization
- dataset metadata for reproducibility

The resulting artifacts are intended for:
- TensorFlow Object Detection API training
- post-training quantization (PTQ)
- quantization-aware training (QAT)
- TensorFlow Lite deployment

## Environment Setup

This notebook targets:
- Python 3.10
- TensorFlow-compatible preprocessing pipeline
- Kaggle notebook runtime

For reproducibility, the exact repository commit is pinned.

In [1]:
!python --version

Python 3.10.10


In [2]:
!pip install -q --no-deps \
  git+https://github.com/frdiener/agri-vision-edge.git

In [3]:
from pathlib import Path
import json

from agri_vision_edge.data import (
    PhenoBench,
    PHENOBENCH_MULTICLASS as DATASET_DEFINITION,
    # PHENOBENCH_WEED_ONLY as DATASET_DEFINITION,
    split_indices,
    build_record,
    build_rep_indices,
    write_label_map,
    export_coco_annotations,
)

SEED = 42

IMAGE_SIZE = 320

DATASET_ROOT = Path(
    "../input/datasets/freimutdiener/phenobench-raw-dataset-v1-1-0/PhenoBench"
)

assert DATASET_ROOT.exists()

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


## Dataset Loading

We load upstream PhenoBench bounding-box annotations and export them into object detection datasets.

In [4]:
train_dataset = PhenoBench(
    root=DATASET_ROOT,
    split="train",
    target_types=[
        "plant_bboxes",
    ],
    ignore_partial=False,
)

val_dataset = PhenoBench(
    root=DATASET_ROOT,
    split="val",
    target_types=[
        "plant_bboxes",
    ],
    ignore_partial=False,
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

Train samples: 1407
Validation samples: 772


## Validation/Test Split

The original validation split is subdivided into:
- validation
- held-out test

using a deterministic random seed.

In [5]:
val_idx, test_idx = split_indices(
    len(val_dataset),
    val_ratio=0.5,
    seed=SEED,
)

with open("val_test_split.json", "w") as f:
    json.dump(
        {
            "val": val_idx,
            "test": test_idx,
        },
        f,
    )

## TFRecord Export

Upstream PhenoBench bounding boxes are exported into:

- TensorFlow Object Detection TFRecords
- COCO evaluation annotations

Generated datasets:

- `train.record`
- `val.record`
- `test.record`
- `true_eval.record`

Images are resized to:

- 320 × 320

In [6]:
train_stats = build_record(
    "train.record",
    train_dataset,
    dataset_definition=DATASET_DEFINITION,
    target_size=IMAGE_SIZE,
)

true_eval_stats = build_record(
    "true_eval.record",
    val_dataset,
    dataset_definition=DATASET_DEFINITION,
    target_size=IMAGE_SIZE,
)

val_stats = build_record(
    "val.record",
    val_dataset,
    dataset_definition=DATASET_DEFINITION,
    indices=val_idx,
    target_size=IMAGE_SIZE,
)

test_stats = build_record(
    "test.record",
    val_dataset,
    dataset_definition=DATASET_DEFINITION,
    indices=test_idx,
    target_size=IMAGE_SIZE,
)

train.record → written: 1407
true_eval.record → written: 772
val.record → written: 386
test.record → written: 386


## COCO Annotation Export

COCO annotations are exported for:

- pycocotools evaluation
- embedded benchmarking
- runtime validation
- framework-independent evaluation

These annotations serve as the canonical evaluation format.

In [7]:
export_coco_annotations(
    "train_annotations.json",
    train_dataset,
    dataset_definition=DATASET_DEFINITION,
)

export_coco_annotations(
    "true_eval_annotations.json",
    val_dataset,
    dataset_definition=DATASET_DEFINITION,
)

export_coco_annotations(
    "val_annotations.json",
    val_dataset,
    dataset_definition=DATASET_DEFINITION,
    indices=val_idx,
)

export_coco_annotations(
    "test_annotations.json",
    val_dataset,
    dataset_definition=DATASET_DEFINITION,
    indices=test_idx,
)

Wrote COCO annotations: train_annotations.json
Wrote COCO annotations: true_eval_annotations.json
Wrote COCO annotations: val_annotations.json
Wrote COCO annotations: test_annotations.json


{'images': 386, 'annotations': 4346, 'categories': 2}

## Representative Dataset for INT8 Quantization

A representative calibration subset is generated for:
- post-training quantization (PTQ)
- quantization-aware training (QAT)

The exported indices ensure:
- deterministic calibration
- reproducible quantization experiments
- consistent deployment benchmarking

In [8]:
rep_indices = build_rep_indices(
    dataset=train_dataset,
    num_samples=200,
    seed=SEED,
)

with open("rep_dataset.json", "w") as f:
    json.dump(rep_indices, f)

## Label Map

TensorFlow Object Detection API requires a label map definition.

In [9]:
write_label_map(
    "label_map.pbtxt",
    dataset_definition=DATASET_DEFINITION,
)

Wrote label map: label_map.pbtxt


## Dataset Metadata

We export metadata required for:
- reproducibility
- downstream training
- deployment consistency

In [10]:
metadata = {

    "dataset_definition": {
        "name": DATASET_DEFINITION.name,
        "categories": DATASET_DEFINITION.categories,
    },
    "image_size": IMAGE_SIZE,
    "train_samples": len(train_dataset),
    "val_samples": len(val_dataset),
    "rep_samples": len(rep_indices),
    "split_seed": SEED,
    "train_stats": train_stats,
    "true_eval_stats": true_eval_stats,
    "val_stats": val_stats,
    "test_stats": test_stats,
}

with open("dataset_metadata.json", "w",) as f:
    json.dump(metadata, f, indent=2,)

## Artifact Verification

Verify all required outputs exist before notebook export.

In [11]:
artifacts = [
    "train.record",
    "val.record",
    "test.record",
    "true_eval.record",

    "train_annotations.json",
    "true_eval_annotations.json",
    "val_annotations.json",
    "test_annotations.json",

    "label_map.pbtxt",
    "rep_dataset.json",
    "val_test_split.json",
    "dataset_metadata.json",
]

missing = [
    p for p in artifacts
    if not Path(p).exists()
]

assert not missing, f"Missing artifacts: {missing}"

print("All artifacts generated successfully.")

All artifacts generated successfully.


In [12]:
for artifact in artifacts:
    print(artifact)

train.record
val.record
test.record
true_eval.record
train_annotations.json
true_eval_annotations.json
val_annotations.json
test_annotations.json
label_map.pbtxt
rep_dataset.json
val_test_split.json
dataset_metadata.json


In [13]:
print(json.dumps(metadata, indent=2))

{
  "dataset_definition": {
    "name": "phenobench_multiclass",
    "categories": [
      {
        "id": 1,
        "name": "crop"
      },
      {
        "id": 2,
        "name": "weed"
      }
    ]
  },
  "image_size": 320,
  "train_samples": 1407,
  "val_samples": 772,
  "rep_samples": 200,
  "split_seed": 42,
  "train_stats": {
    "written": 1407,
    "target_size": 320
  },
  "true_eval_stats": {
    "written": 772,
    "target_size": 320
  },
  "val_stats": {
    "written": 386,
    "target_size": 320
  },
  "test_stats": {
    "written": 386,
    "target_size": 320
  }
}


## Notes

This export pipeline provides a reproducible bridge between:

- PhenoBench bounding-box annotations
- TensorFlow object detection training
- COCO-native evaluation
- edge deployment workflows

Generated artifacts support:

- TensorFlow Object Detection API
- TensorFlow Lite conversion
- post-training quantization (PTQ)
- quantization-aware training (QAT)
- embedded benchmarking
- pycocotools evaluation

COCO annotations are exported as the canonical
framework-independent evaluation format.